# Qugeister - HNN Composer QNN学習

HNN Composerで設計したカスタムアーキテクチャをそのまま学習します。

## ワークフロー
1. QuAic HNN Composerで`config.json`をエクスポート
2. このノートブックで学習を実行
3. `weights.pth`をダウンロードしてQuAicに提出

**特徴**: HNN Composerで設計したレイヤー構造がそのまま反映されます。

## 1. 環境セットアップ

In [ ]:
# 必要なライブラリをインストール
!pip install -q pennylane torch numpy tqdm matplotlib

In [ ]:
import json
import pickle
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
import pennylane as qml
from tqdm.auto import tqdm
from google.colab import files

# デバイス設定
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

## 2. 設定ファイルのアップロード

QuAic HNN Composerからエクスポートした`config.json`をアップロードしてください。

In [ ]:
# config.jsonをアップロード
print('config.jsonをアップロードしてください...')
uploaded = files.upload()

# 設定を読み込み
config_filename = list(uploaded.keys())[0]
with open(config_filename, 'r') as f:
    config = json.load(f)

# モデル名を取得（両形式に対応）
model_name = config.get('qugeister_metadata', {}).get('model_name') or config.get('model_name', 'unnamed')
description = config.get('description', 'N/A')

print(f'\n設定を読み込みました: {model_name}')
print(f'説明: {description}')

## 3. HNN設定のパース

HNN Composerのグラフ構造から、レイヤー構成を抽出します。

In [ ]:
def parse_hnn_config(config):
    """HNN Composer設定をパースしてアーキテクチャを抽出
    
    対応形式:
    1. Qugeisterエクスポート形式 (architecture.quantum)
    2. Unified Trainer Config形式 (module_04_neural_networks)
    3. 生グラフ形式 (network.nodes)
    
    Returns:
        dict: {
            'n_qubits': int,
            'n_layers': int,
            'pre_layers': list of layer configs,
            'post_layers': list of layer configs,
            'training': training config
        }
    """
    
    # デバッグ: 設定ファイルのキーを表示
    print(f'設定ファイルのトップレベルキー: {list(config.keys())}')
    
    # 形式1: Qugeisterエクスポート形式 (architecture.quantum)
    if 'architecture' in config and 'quantum' in config.get('architecture', {}):
        arch = config['architecture']
        quantum = arch['quantum']
        
        n_qubits = quantum.get('n_qubits', 4)
        n_layers = quantum.get('n_layers', 2)
        
        pre_layers = []
        for layer in arch.get('classical_pre', []):
            if layer.get('type') == 'linear':
                pre_layers.append({
                    'type': 'linear',
                    'out_features': layer.get('out_features', n_qubits)
                })
        if not pre_layers:
            pre_layers = [{'type': 'linear', 'out_features': n_qubits}]
        
        post_layers = []
        for layer in arch.get('classical_post', []):
            if layer.get('type') == 'linear':
                post_layers.append({
                    'type': 'linear',
                    'out_features': layer.get('out_features', 16)
                })
        if not post_layers:
            post_layers = [{'type': 'linear', 'out_features': 16}]
        
        print(f'\n=== HNN設定 (Qugeisterエクスポート形式) ===')
        print(f'量子ビット数: {n_qubits}')
        print(f'量子レイヤー数: {n_layers}')
        
        return {
            'n_qubits': n_qubits,
            'n_layers': n_layers,
            'pre_layers': pre_layers,
            'post_layers': post_layers,
            'training': config.get('training', {})
        }
    
    # 形式2: Unified Trainer Config形式 (module_04_neural_networks または module_02_neural_networks)
    neural_module = config.get('module_04_neural_networks') or config.get('module_02_neural_networks')
    if neural_module:
        module_config = neural_module.get('config', neural_module)
        
        # quantum_config を探す
        quantum_config = None
        graph_structure = None
        
        if 'color_estimation' in module_config:
            ce = module_config['color_estimation']
            quantum_config = ce.get('quantum_config')
            graph_structure = ce.get('graph_structure')
        elif 'quantum_config' in module_config:
            quantum_config = module_config['quantum_config']
            graph_structure = module_config.get('graph_structure')
        
        if quantum_config:
            n_qubits = quantum_config.get('n_qubits', 4)
            n_layers = quantum_config.get('n_layers', 2)
            
            # グラフ構造から前後処理レイヤーを抽出
            pre_layers = [{'type': 'linear', 'out_features': n_qubits}]
            post_layers = [{'type': 'linear', 'out_features': 16}]
            
            if graph_structure:
                nodes = graph_structure.get('nodes', [])
                for node in nodes:
                    if node.get('type') == 'dense':
                        out_f = node.get('data', {}).get('out_features')
                        if out_f:
                            # 後で順序を決定
                            pass
            
            print(f'\n=== HNN設定 (Unified Trainer Config形式) ===')
            print(f'量子ビット数: {n_qubits}')
            print(f'量子レイヤー数: {n_layers}')
            
            return {
                'n_qubits': n_qubits,
                'n_layers': n_layers,
                'pre_layers': pre_layers,
                'post_layers': post_layers,
                'training': config.get('training', {})
            }
    
    # 形式3: 生グラフ形式 (network.nodes)
    network = config.get('network', {})
    nodes = network.get('nodes', [])
    edges = network.get('edges', [])
    
    if not nodes:
        raise ValueError(
            f'設定ファイルの形式が認識できません。\n'
            f'トップレベルキー: {list(config.keys())}\n'
            f'\n対応形式:\n'
            f'  1. architecture.quantum (Qugeister export)\n'
            f'  2. module_04_neural_networks (Unified Trainer Config)\n'
            f'  3. network.nodes (Raw graph)\n'
            f'\nHNN Composerから正しくエクスポートしてください。'
        )
    
    # ノード・エッジマップ作成
    node_map = {node['id']: node for node in nodes}
    outgoing = {}
    incoming = {}
    for edge in edges:
        src, tgt = edge.get('source'), edge.get('target')
        if src and tgt:
            outgoing.setdefault(src, []).append(tgt)
            incoming.setdefault(tgt, []).append(src)
    
    # 量子ノードを探す
    quantum_node = None
    for node in nodes:
        if node.get('type') == 'quantum':
            quantum_node = node
            break
    
    if not quantum_node:
        raise ValueError('量子ノードが見つかりません')
    
    q_data = quantum_node.get('data', {})
    q_circuit = q_data.get('qCircuit', {})
    n_qubits = q_data.get('n_qubits', 4)
    n_layers = q_circuit.get('n_layers', q_circuit.get('ansatzRepeats', 2))
    
    quantum_id = quantum_node['id']
    
    # 前処理レイヤー
    pre_layers = []
    visited = set()
    def find_pre_dense(node_id):
        if node_id in visited or node_id == quantum_id:
            return
        visited.add(node_id)
        node = node_map.get(node_id)
        if node and node.get('type') == 'dense':
            out_f = node.get('data', {}).get('out_features', n_qubits)
            pre_layers.append({'type': 'linear', 'out_features': out_f})
        for src in incoming.get(node_id, []):
            find_pre_dense(src)
    for src in incoming.get(quantum_id, []):
        find_pre_dense(src)
    
    # 後処理レイヤー
    post_layers = []
    visited = set()
    def find_post_dense(node_id):
        if node_id in visited or node_id == quantum_id:
            return
        visited.add(node_id)
        node = node_map.get(node_id)
        if node and node.get('type') == 'dense':
            out_f = node.get('data', {}).get('out_features', 16)
            post_layers.append({'type': 'linear', 'out_features': out_f})
        for tgt in outgoing.get(node_id, []):
            find_post_dense(tgt)
    for tgt in outgoing.get(quantum_id, []):
        find_post_dense(tgt)
    
    if not pre_layers:
        pre_layers = [{'type': 'linear', 'out_features': n_qubits}]
    if not post_layers:
        post_layers = [{'type': 'linear', 'out_features': 16}]
    
    print(f'\n=== HNN設定 (グラフ形式) ===')
    print(f'量子ビット数: {n_qubits}')
    print(f'量子レイヤー数: {n_layers}')
    
    return {
        'n_qubits': n_qubits,
        'n_layers': n_layers,
        'pre_layers': pre_layers,
        'post_layers': post_layers,
        'training': config.get('training', {})
    }

hnn_config = parse_hnn_config(config)

## 4. HNNモデルの構築

HNN Composerの設計に基づいて動的にモデルを構築します。

In [ ]:
class QuantumLayer(nn.Module):
    """PennyLane量子レイヤー
    
    パラメータ: weights [n_layers, n_qubits, 2] (RY, RZ)
    """

    def __init__(self, n_qubits, n_layers):
        super().__init__()
        self.n_qubits = n_qubits
        self.n_layers = n_layers

        # 量子デバイス
        self.dev = qml.device('default.qubit', wires=n_qubits)

        # 量子パラメータ [n_layers, n_qubits, 2]
        self.weights = nn.Parameter(
            torch.randn(n_layers, n_qubits, 2) * 0.1
        )

        @qml.qnode(self.dev, interface='torch', diff_method='backprop')
        def circuit(inputs, weights):
            # 入力埋め込み (RY)
            for i in range(n_qubits):
                qml.RY(inputs[i], wires=i)

            # Variationalレイヤー
            for layer in range(n_layers):
                # エンタングル
                for i in range(n_qubits - 1):
                    qml.CNOT(wires=[i, i + 1])

                # パラメータ回転 (RY, RZ)
                for i in range(n_qubits):
                    qml.RY(weights[layer, i, 0], wires=i)
                    qml.RZ(weights[layer, i, 1], wires=i)

            return [qml.expval(qml.PauliZ(i)) for i in range(n_qubits)]

        self.circuit = circuit

    def forward(self, x):
        batch_size = x.shape[0]
        results = []
        for i in range(batch_size):
            result = self.circuit(x[i], self.weights)
            results.append(torch.stack(result))
        return torch.stack(results)

In [ ]:
class HNNColorEstimator(nn.Module):
    """HNN Composer設計に基づく色推定モデル
    
    構造:
    - pre_layers: 入力 (448) -> 量子入力 (n_qubits)
    - quantum_layer: 量子回路
    - post_layers: 量子出力 -> 色推定 (16 = 8駒 x 2)
    
    出力形式: HNN形式 (pre_layers.*, quantum_layer.weights, post_layers.*)
    """

    def __init__(self, hnn_config, input_size=448, n_pieces=8):
        super().__init__()
        self.n_pieces = n_pieces
        self.n_qubits = hnn_config['n_qubits']
        self.n_layers = hnn_config['n_layers']
        
        # 前処理レイヤーを構築
        pre_layers_list = []
        current_size = input_size
        
        for layer_cfg in hnn_config['pre_layers']:
            if layer_cfg['type'] == 'linear':
                out_size = layer_cfg['out_features']
                pre_layers_list.append(nn.Linear(current_size, out_size))
                current_size = out_size
        
        self.pre_layers = nn.Sequential(*pre_layers_list)
        
        # 量子レイヤー
        self.quantum_layer = QuantumLayer(
            n_qubits=self.n_qubits,
            n_layers=self.n_layers
        )
        
        # 後処理レイヤーを構築
        post_layers_list = []
        current_size = self.n_qubits
        
        for layer_cfg in hnn_config['post_layers']:
            if layer_cfg['type'] == 'linear':
                out_size = layer_cfg['out_features']
                post_layers_list.append(nn.Linear(current_size, out_size))
                current_size = out_size
        
        self.post_layers = nn.Sequential(*post_layers_list)

    def forward(self, x):
        batch_size = x.shape[0]
        
        # 前処理
        x = self.pre_layers(x)
        
        # 量子層（PennyLaneはDoubleを出力するのでFloatに変換）
        x = self.quantum_layer(x)
        x = x.float()  # Double -> Float変換
        
        # 後処理
        x = self.post_layers(x)  # [batch, 16]
        x = x.view(batch_size, self.n_pieces, 2)  # [batch, 8, 2]
        
        return x


# モデルを作成
model = HNNColorEstimator(hnn_config).to(device)

print(f'\n=== モデル構造 ===')
print(f'パラメータ数: {sum(p.numel() for p in model.parameters()):,}')
print(f'\nstate_dict キー:')
for key, value in model.state_dict().items():
    print(f'  {key}: {value.shape}')

## 5. 棋譜データのアップロード

QuAicの「コンペティション > データ」から棋譜データをダウンロードしてアップロードしてください。

In [ ]:
# 棋譜データをアップロード
print('棋譜データ(.pkl)をアップロードしてください...')
uploaded_data = files.upload()
TRAJECTORY_FILE = list(uploaded_data.keys())[0]

# データを読み込み
with open(TRAJECTORY_FILE, 'rb') as f:
    trajectory_data = pickle.load(f)

print(f'読み込んだ試合数: {len(trajectory_data)}')

In [ ]:
def prepare_data(trajectory_data, train_ratio=0.8):
    """棋譜データから学習用データを準備"""
    X_list = []
    y_list = []

    for traj in trajectory_data:
        # Player A
        if 'states_A' in traj and 'true_colors_A' in traj:
            for state, colors in zip(traj['states_A'], traj['true_colors_A']):
                state = np.array(state).flatten()
                colors = np.array(colors)
                if state.shape[0] == 448 and colors.shape[0] == 8:
                    X_list.append(state)
                    y_list.append(colors)

        # Player B
        if 'states_B' in traj and 'true_colors_B' in traj:
            for state, colors in zip(traj['states_B'], traj['true_colors_B']):
                state = np.array(state).flatten()
                colors = np.array(colors)
                if state.shape[0] == 448 and colors.shape[0] == 8:
                    X_list.append(state)
                    y_list.append(colors)

    X = np.array(X_list, dtype=np.float32)
    y = np.array(y_list, dtype=np.int64)

    # シャッフル
    indices = np.random.permutation(len(X))
    X, y = X[indices], y[indices]

    # 分割
    split_idx = int(len(X) * train_ratio)
    X_train, X_val = X[:split_idx], X[split_idx:]
    y_train, y_val = y[:split_idx], y[split_idx:]

    print(f'学習データ: {len(X_train)} サンプル')
    print(f'検証データ: {len(X_val)} サンプル')

    return X_train, y_train, X_val, y_val

X_train, y_train, X_val, y_val = prepare_data(trajectory_data)

# DataLoader作成
train_dataset = TensorDataset(
    torch.tensor(X_train),
    torch.tensor(y_train)
)
val_dataset = TensorDataset(
    torch.tensor(X_val),
    torch.tensor(y_val)
)

batch_size = hnn_config['training'].get('batch_size', 32)
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size)

## 6. 学習

In [ ]:
# 学習設定
training_config = hnn_config['training']
epochs = training_config.get('epochs', 50)
learning_rate = training_config.get('learning_rate', 0.001)

print(f'エポック数: {epochs}')
print(f'学習率: {learning_rate}')
print(f'バッチサイズ: {batch_size}')

# オプティマイザと損失関数
optimizer = optim.Adam(model.parameters(), lr=learning_rate)
criterion = nn.CrossEntropyLoss()

In [ ]:
def train_epoch(model, loader, optimizer, criterion, device):
    model.train()
    total_loss = 0
    correct = 0
    total = 0

    for X_batch, y_batch in loader:
        X_batch = X_batch.to(device)
        y_batch = y_batch.to(device)

        optimizer.zero_grad()
        outputs = model(X_batch)  # [batch, 8, 2]

        # 損失計算
        loss = criterion(outputs.view(-1, 2), y_batch.view(-1))

        loss.backward()
        optimizer.step()

        total_loss += loss.item()

        # 精度計算
        preds = outputs.argmax(dim=-1)
        correct += (preds == y_batch).sum().item()
        total += y_batch.numel()

    return total_loss / len(loader), correct / total


def validate(model, loader, criterion, device):
    model.eval()
    total_loss = 0
    correct = 0
    total = 0

    with torch.no_grad():
        for X_batch, y_batch in loader:
            X_batch = X_batch.to(device)
            y_batch = y_batch.to(device)

            outputs = model(X_batch)
            loss = criterion(outputs.view(-1, 2), y_batch.view(-1))

            total_loss += loss.item()

            preds = outputs.argmax(dim=-1)
            correct += (preds == y_batch).sum().item()
            total += y_batch.numel()

    return total_loss / len(loader), correct / total

In [ ]:
# 学習ループ
best_val_acc = 0
history = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': []}

print('学習を開始します...\n')

for epoch in tqdm(range(epochs), desc='Training'):
    train_loss, train_acc = train_epoch(model, train_loader, optimizer, criterion, device)
    val_loss, val_acc = validate(model, val_loader, criterion, device)

    history['train_loss'].append(train_loss)
    history['train_acc'].append(train_acc)
    history['val_loss'].append(val_loss)
    history['val_acc'].append(val_acc)

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(model.state_dict(), 'best_model.pth')

    if (epoch + 1) % 10 == 0:
        print(f'Epoch {epoch+1}/{epochs}')
        print(f'  Train Loss: {train_loss:.4f}, Acc: {train_acc:.4f}')
        print(f'  Val Loss: {val_loss:.4f}, Acc: {val_acc:.4f}')

print(f'\n学習完了! ベスト検証精度: {best_val_acc:.4f}')

## 7. 学習曲線の可視化

In [ ]:
import matplotlib.pyplot as plt

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

# 損失
ax1.plot(history['train_loss'], label='Train')
ax1.plot(history['val_loss'], label='Validation')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Loss')
ax1.set_title('Training Loss')
ax1.legend()
ax1.grid(True)

# 精度
ax2.plot(history['train_acc'], label='Train')
ax2.plot(history['val_acc'], label='Validation')
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Accuracy')
ax2.set_title('Training Accuracy')
ax2.legend()
ax2.grid(True)

plt.tight_layout()
plt.savefig('training_history.png', dpi=150)
plt.show()

## 8. モデルのエクスポート

学習済みモデルをダウンロードします。

**出力形式**: HNN形式 (`pre_layers.*`, `quantum_layer.weights`, `post_layers.*`)

In [ ]:
# ベストモデルを読み込み
model.load_state_dict(torch.load('best_model.pth'))

# エクスポート（HNN形式のまま）
model_name = config.get('qugeister_metadata', {}).get('model_name') or config.get('model_name', 'hnn_model')
export_filename = f'{model_name}_weights.pth'

torch.save(model.state_dict(), export_filename)
print(f'モデルを保存しました: {export_filename}')
print(f'\nstate_dict キー:')
for key, value in model.state_dict().items():
    print(f'  {key}: {value.shape}')

In [ ]:
# ダウンロード
print('ダウンロードを開始...')
files.download(export_filename)
print('\nダウンロード完了!')
print('\n次のステップ:')
print('1. QuAic (https://quaic.up.railway.app) にアクセス')
print('2. 「コンペティション > モデル」に移動')
print('3. ダウンロードした .pth ファイルをアップロード')
print('\n※ HNN形式 (pre_layers.*, quantum_layer.*, post_layers.*) はQuAicで自動認識されます')